In [1]:
# ==============================================================
# 🇲🇦 PIPELINE COMPLET — Qwen2.5-1.5B Finetuning sur Dataset Darija
# Prétraitement → Finetuning → Test
# Compatible: Google Colab (GPU T4 / A100)
# Dataset : /content/drive/MyDrive/fine tuning/dataset.csv
# Sortie  : /content/drive/MyDrive/fine tuning/
# ==============================================================


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — Installation des dépendances                   ║
# ║  ⚠️ Exécuter UNE SEULE FOIS puis redémarrer le runtime      ║
# ╚══════════════════════════════════════════════════════════════╝

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install pandas datasets transformers sentencepiece


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — PRÉTRAITEMENT DU DATASET                       ║
# ╚══════════════════════════════════════════════════════════════╝

import os
import re
import pandas as pd
from datasets import Dataset
from google.colab import drive

# ── Monter Google Drive ──────────────────────────────────────
drive.mount('/content/drive')

# ── Chemins (espace dans le nom → guillemets + raw path) ─────
BASE_DIR     = "/content/drive/MyDrive/fine tuning"
DATASET_PATH = f"{BASE_DIR}/dataset.csv"
CLEAN_PATH   = f"{BASE_DIR}/dataset_clean.csv"
REPORT_PATH  = f"{BASE_DIR}/preprocessing_report.txt"

os.makedirs(BASE_DIR, exist_ok=True)

# ── Paramètres de filtrage ───────────────────────────────────
MIN_LENGTH = 50
MAX_LENGTH = 8000

print("=" * 60)
print("📂 CHARGEMENT DU DATASET")
print("=" * 60)

df = pd.read_csv(DATASET_PATH)
print(f"✅ Lignes totales  : {len(df)}")
print(f"   Colonnes        : {list(df.columns)}")
print(f"\n📌 Aperçu :")
print(df.head(3).to_string())

# ── Analyse initiale ─────────────────────────────────────────
print("\n" + "=" * 60)
print("🔍 ANALYSE INITIALE")
print("=" * 60)

print(f"\n📊 Valeurs nulles :")
print(df.isnull().sum())

if 'language' in df.columns:
    print(f"\n📊 Langues :")
    print(df['language'].value_counts())

if 'topic' in df.columns:
    print(f"\n📊 Topics (top 10) :")
    print(df['topic'].value_counts().head(10))

df['text_length'] = df['text'].astype(str).apply(len)
print(f"\n📊 Longueurs textes (chars) :")
print(f"   Min    : {df['text_length'].min()}")
print(f"   Max    : {df['text_length'].max()}")
print(f"   Moyenne: {df['text_length'].mean():.0f}")
print(f"   Médiane: {df['text_length'].median():.0f}")

# ── Nettoyage ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🧹 NETTOYAGE")
print("=" * 60)

# 1. Doublons
before = len(df)
df = df.drop_duplicates(subset=['text'])
print(f"[1] Doublons supprimés          : {before - len(df)}")

# 2. Lignes nulles / vides
before = len(df)
df = df.dropna(subset=['text'])
df = df[df['text'].astype(str).str.strip() != '']
print(f"[2] Lignes null/vides supprimées: {before - len(df)}")

# 3. Vérification format ChatML
def has_valid_chatml(text: str) -> bool:
    return all(tag in text for tag in [
        '<|im_start|>', '<|im_end|>',
        '<|im_start|>system',
        '<|im_start|>user',
        '<|im_start|>assistant'
    ])

before = len(df)
df['valid_chatml'] = df['text'].astype(str).apply(has_valid_chatml)
invalid_count = (~df['valid_chatml']).sum()
print(f"[3] Format ChatML invalide      : {invalid_count} lignes ignorées")

# ⚠️ Si le dataset N'utilise PAS le format ChatML, commenter les 2 lignes suivantes
df = df[df['valid_chatml']].drop(columns=['valid_chatml'])
print(f"    Après filtre ChatML         : {len(df)} lignes")

# 4. Nettoyage du texte
def clean_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text.strip()

df['text'] = df['text'].astype(str).apply(clean_text)
print(f"[4] Nettoyage caractères        : ✅")

# 5. Filtre longueur
before = len(df)
df['text_length'] = df['text'].apply(len)
df = df[(df['text_length'] >= MIN_LENGTH) & (df['text_length'] <= MAX_LENGTH)]
print(f"[5] Filtre longueur [{MIN_LENGTH}-{MAX_LENGTH}]    : {before - len(df)} supprimés → {len(df)} restants")

# 6. Réponse assistant non vide
def has_non_empty_assistant(text: str) -> bool:
    parts = text.split('<|im_start|>assistant')
    if len(parts) < 2:
        return False
    return len(parts[-1].replace('<|im_end|>', '').strip()) > 10

before = len(df)
df = df[df['text'].apply(has_non_empty_assistant)]
print(f"[6] Réponses assistant vides    : {before - len(df)} supprimés → {len(df)} restants")

# 7. Formatage final
def format_for_training(text: str) -> str:
    text = text.rstrip()
    if not text.endswith('<|im_end|>'):
        text += '<|im_end|>'
    return text

df['text'] = df['text'].apply(format_for_training)

# 8. Métadonnées
df['num_turns']        = df['text'].apply(lambda t: t.count('<|im_start|>user'))
df['text_length']      = df['text'].apply(len)
df['num_tokens_approx'] = df['text_length'] // 4

# ── Stats finales ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 STATISTIQUES FINALES")
print("=" * 60)
print(f"✅ Exemples       : {len(df)}")
print(f"   Tokens totaux  : ~{df['num_tokens_approx'].sum():,}")
print(f"   Tokens/exemple : ~{df['num_tokens_approx'].mean():.0f}")

if 'language' in df.columns:
    print(f"\n🌍 Langues :")
    print(df['language'].value_counts().to_string())

print(f"\n💬 Tours de conversation :")
print(df['num_turns'].value_counts().sort_index().to_string())

# ── Sauvegarde ───────────────────────────────────────────────
cols_to_save = [c for c in ['text', 'topic', 'language', 'source_file'] if c in df.columns]
df[cols_to_save].to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
print(f"\n💾 Dataset propre sauvegardé : {CLEAN_PATH}")

# Rapport
report = [
    "=" * 60,
    "RAPPORT PRÉTRAITEMENT — QWEN2.5-1.5B DARIJA",
    "=" * 60,
    f"Source    : {DATASET_PATH}",
    f"Sortie    : {CLEAN_PATH}",
    f"Exemples  : {len(df)}",
    f"Long. moy.: {df['text_length'].mean():.0f} chars",
    f"Tokens tot: ~{df['num_tokens_approx'].sum():,}",
]
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report))
print(f"✅ Rapport sauvegardé        : {REPORT_PATH}")

print("\n✅ PRÉTRAITEMENT TERMINÉ — passe à la Cellule 2 !")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7u1o9jhb/unsloth_2bd8fbaf0f3744f1a2d7bf31a0e5cd7b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7u1o9jhb/unsloth_2bd8fbaf0f3744f1a2d7bf31a0e5cd7b
  Resolved https://github.com/unslothai/unsloth.git to commit 50cccfd55ea9e0979a51d22d23acd092f50647aa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.2 MB/s eta 0:00:00

In [2]:
# Cellule de diagnostic — à exécuter après Cellule 1
df_original = pd.read_csv(f"{BASE_DIR}/dataset.csv")
df_clean    = pd.read_csv(f"{BASE_DIR}/dataset_clean.csv")

print("=== DIAGNOSTIC PERTE PAR LANGUE ===")
for lang in ['moroccan_darija', 'arabic', 'french', 'english']:
    orig  = len(df_original[df_original['language'] == lang])
    clean = len(df_clean[df_clean['language'] == lang])
    perte = orig - clean
    print(f"{lang:20s}: {orig} → {clean}  (perdu: {perte})")

print("\n=== EXEMPLES SANS LANGUE (language=NaN) ===")
sans_langue = df_clean[df_clean['language'].isna()]
print(f"Nombre : {len(sans_langue)}")
print(sans_langue['source_file'].value_counts())

=== DIAGNOSTIC PERTE PAR LANGUE ===
moroccan_darija     : 600 → 356  (perdu: 244)
arabic              : 600 → 337  (perdu: 263)
french              : 600 → 331  (perdu: 269)
english             : 600 → 600  (perdu: 0)

=== EXEMPLES SANS LANGUE (language=NaN) ===
Nombre : 6385
source_file
Français.jsonl               1964
DARIJA.jsonl                 1922
ENGLISH_data.jsonl            984
train.jsonl                   911
arabic_chatbot_data.jsonl     505
eval.jsonl                     99
Name: count, dtype: int64


In [8]:
# Cellule CORRECTION — Renseigner les langues manquantes
import pandas as pd

BASE_DIR   = "/content/drive/MyDrive/fine tuning"
CLEAN_PATH = f"{BASE_DIR}/dataset_clean.csv"

df = pd.read_csv(CLEAN_PATH)

print(f"Avant correction : {df['language'].isna().sum()} NaN")

# Mapping source_file → langue
source_to_lang = {
    'Français.jsonl'              : 'french',
    'DARIJA.jsonl'                : 'moroccan_darija',
    'ENGLISH_data.jsonl'          : 'english',
    'arabic_chatbot_data.jsonl'   : 'arabic',
}

df['language'] = df.apply(
    lambda row: source_to_lang.get(row['source_file'], row['language'])
    if pd.isna(row['language']) else row['language'],
    axis=1
)

print(f"Après correction : {df['language'].isna().sum()} NaN")
print(f"\n📊 Distribution finale des langues :")
print(df['language'].value_counts())

# Sauvegarder
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
print(f"\n✅ Dataset mis à jour : {CLEAN_PATH}")

Avant correction : 0 NaN
Après correction : 0 NaN

📊 Distribution finale des langues :
language
french             2295
moroccan_darija    2278
english            1584
unknown            1010
arabic              842
Name: count, dtype: int64

✅ Dataset mis à jour : /content/drive/MyDrive/fine tuning/dataset_clean.csv


In [9]:
df.shape

(8009, 4)

In [10]:
# Cellule RÉÉQUILIBRAGE — à exécuter AVANT la Cellule 2
import pandas as pd

BASE_DIR   = "/content/drive/MyDrive/fine tuning"
CLEAN_PATH = f"{BASE_DIR}/dataset_clean.csv"

df = pd.read_csv(CLEAN_PATH)

# 1. Supprimer les 'unknown'
df = df[df['language'] != 'unknown']
print(f"Après suppression unknown : {len(df)} exemples")

# 2. Rééquilibrer par langue (max = langue la moins représentée)
TARGET = df['language'].value_counts().min()
print(f"Cible par langue : {TARGET} exemples")

df_balanced = (
    df.groupby('language', group_keys=False)
      .apply(lambda x: x.sample(n=TARGET, random_state=42))
      .reset_index(drop=True)
)

print(f"\n📊 Distribution équilibrée :")
print(df_balanced['language'].value_counts())
print(f"\n✅ Total : {len(df_balanced)} exemples")

# Sauvegarder
BALANCED_PATH = f"{BASE_DIR}/dataset_balanced.csv"
df_balanced.to_csv(BALANCED_PATH, index=False, encoding='utf-8-sig')
print(f"💾 Sauvegardé : {BALANCED_PATH}")

Après suppression unknown : 6999 exemples
Cible par langue : 842 exemples

📊 Distribution équilibrée :
language
arabic             842
english            842
french             842
moroccan_darija    842
Name: count, dtype: int64

✅ Total : 3368 exemples
💾 Sauvegardé : /content/drive/MyDrive/fine tuning/dataset_balanced.csv


/tmp/ipykernel_6174/3807239785.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=TARGET, random_state=42))


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — FINETUNING Qwen2.5-1.5B-Instruct              ║
# ║  ✅ Dataset équilibré par langue (sans 'unknown')            ║
# ╚══════════════════════════════════════════════════════════════╝

import unsloth   # ← TOUJOURS en premier
import os, torch, pandas as pd
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from google.colab import drive

drive.mount('/content/drive')

# ── Configuration ────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

BASE_DIR     = "/content/drive/MyDrive/fine tuning"
DATASET_PATH = f"{BASE_DIR}/dataset_clean.csv"
OUTPUT_DIR   = f"{BASE_DIR}/lora_adapters"
MERGED_DIR   = f"{BASE_DIR}/model_merged_final"
LOGS_DIR     = f"{BASE_DIR}/training_logs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────
NUM_EPOCHS       = 3
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE    = 2e-4
WARMUP_STEPS     = 10
MAX_STEPS        = -1
SAVE_STEPS       = 50
LOGGING_STEPS    = 10

# ── LoRA ─────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Chargement modèle ────────────────────────────────────────
print("=" * 60)
print(f"📦 Chargement : {MODEL_NAME}")
print("=" * 60)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)

print(f"✅ Modèle chargé avec LoRA")
model.print_trainable_parameters()

# ╔══════════════════════════════════════════════════════════════╗
# ║  RÉÉQUILIBRAGE DU DATASET                                   ║
# ╚══════════════════════════════════════════════════════════════╝
print(f"\n📂 Chargement dataset : {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
print(f"✅ Exemples bruts : {len(df)}")

print(f"\n📊 Distribution AVANT rééquilibrage :")
print(df['language'].value_counts().to_string())

# ── Étape 1 : Supprimer les 'unknown' ────────────────────────
before = len(df)
df = df[df['language'] != 'unknown']
df = df[df['language'].notna()]
print(f"\n[1] Suppression 'unknown'/NaN : {before - len(df)} supprimés → {len(df)} restants")

# ── Étape 2 : Rééquilibrage par langue ───────────────────────
#     On prend le MIN des langues pour équilibrer
TARGET_PER_LANG = df['language'].value_counts().min()
print(f"[2] Cible par langue          : {TARGET_PER_LANG} exemples")

df_balanced = (
    df.groupby('language', group_keys=False)
      .apply(lambda x: x.sample(n=TARGET_PER_LANG, random_state=42))
      .reset_index(drop=True)
      .sample(frac=1, random_state=42)   # shuffle global après équilibrage
      .reset_index(drop=True)
)

print(f"\n📊 Distribution APRÈS rééquilibrage :")
print(df_balanced['language'].value_counts().to_string())
print(f"\n✅ Dataset équilibré final : {len(df_balanced)} exemples")
print(f"   Tokens estimés         : ~{(df_balanced['text'].str.len().sum() // 4):,}")

# Sauvegarder le dataset équilibré (utile pour référence future)
BALANCED_PATH = f"{BASE_DIR}/dataset_balanced.csv"
df_balanced.to_csv(BALANCED_PATH, index=False, encoding='utf-8-sig')
print(f"💾 Dataset équilibré sauvegardé : {BALANCED_PATH}")

# ── Formatage EOS ─────────────────────────────────────────────
EOS_TOKEN = tokenizer.eos_token
print(f"\n🔤 EOS token : {repr(EOS_TOKEN)}")

def add_eos(example):
    text = str(example['text'])
    if not text.endswith(EOS_TOKEN):
        text += EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_pandas(df_balanced[['text']].reset_index(drop=True))
dataset = dataset.map(add_eos)

# Split 90 / 10  (stratifié par langue pour garder l'équilibre)
split         = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f"\n✅ Split dataset :")
print(f"   Train : {len(train_dataset)} exemples")
print(f"   Eval  : {len(eval_dataset)}  exemples")

# ── TrainingArguments ─────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    warmup_steps                = WARMUP_STEPS,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = LOGGING_STEPS,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    eval_strategy               = "steps",
    eval_steps                  = SAVE_STEPS,
    load_best_model_at_end      = True,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    seed                        = 42,
    report_to                   = "none",
    logging_dir                 = LOGS_DIR,
    dataloader_num_workers      = 0,
)

# ── SFTTrainer ────────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args               = training_args,
)

# ── Entraînement ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("🚀 LANCEMENT DE L'ENTRAÎNEMENT")
print("=" * 60)

gpu_stats        = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory       = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU  : {gpu_stats.name}")
print(f"VRAM : {max_memory} GB | Utilisé avant entraînement : {start_gpu_memory} GB")

trainer_stats  = trainer.train()
end_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

print(f"\n✅ ENTRAÎNEMENT TERMINÉ !")
print(f"   Durée       : {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"   Loss finale : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   VRAM max    : {end_gpu_memory} GB / {max_memory} GB")

# ── Sauvegarde adapters LoRA ──────────────────────────────────
print(f"\n💾 Sauvegarde LoRA adapters → {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Adapters LoRA sauvegardés !")

# ── Merge et sauvegarde finale ────────────────────────────────
print(f"\n💾 Merge modèle complet (float16) → {MERGED_DIR}")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("✅ Modèle mergé sauvegardé — utilise ce dossier pour l'inférence !")

print(f"\n📁 Résumé des dossiers :")
print(f"   Dataset équilibré : {BALANCED_PATH}")
print(f"   LoRA adapters     : {OUTPUT_DIR}")
print(f"   Modèle final      : {MERGED_DIR}")
print(f"   Logs              : {LOGS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Chargement : Qwen/Qwen2.5-1.5B-Instruct
==((====))==  Unsloth 2026.3.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Modèle chargé avec LoRA
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

📂 Chargement dataset : /content/drive/MyDrive/fine tuning/dataset_clean.csv
✅ Exemples bruts : 8009

📊 Distribution AVANT rééquilibrage :
language
french             2295
moroccan_darija    2278
english            1584
unknown            1010
arabic              842

[1] Suppression 'unknown'/NaN : 1010 supprimés → 6999 restants
[2] Cible par langue          : 842 exemples

📊 Distribution APRÈS rééquilibrage :
language
moroccan_darija    842
english            842
french             842
arabic             842

✅ Dataset équilibré final : 3368 exemples
   Tokens estimés         : ~664,580
💾 Dataset équilibré sauvegardé : /content/drive/MyDrive/fine tuning/dataset_balanced.csv

🔤 EOS token : '<|im_end|>'


/tmp/ipykernel_6174/3727389173.py:100: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=TARGET_PER_LANG, random_state=42))


Map:   0%|          | 0/3368 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:05<00:00, 65.21s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:08<00:00, 128.81s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/fine tuning/model_merged_final`
✅ Modèle mergé sauvegardé — utilise ce dossier pour l'inférence !

📁 Résumé des dossiers :
   Dataset équilibré : /content/drive/MyDrive/fine tuning/dataset_balanced.csv
   LoRA adapters     : /content/drive/MyDrive/fine tuning/lora_adapters
   Modèle final      : /content/drive/MyDrive/fine tuning/model_merged_final
   Logs              : /content/drive/MyDrive/fine tuning/training_logs


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — TEST DU MODÈLE ENTRAÎNÉ                       ║
# ║  ✅ Adapté au dataset équilibré (arabic/darija/fr/en)        ║
# ╚══════════════════════════════════════════════════════════════╝

import unsloth
import torch
import pandas as pd
import random
from unsloth import FastLanguageModel
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR       = "/content/drive/MyDrive/fine tuning"
MERGED_DIR     = f"{BASE_DIR}/model_merged_final"
BALANCED_PATH  = f"{BASE_DIR}/dataset_balanced.csv"
MAX_SEQ_LENGTH = 2048

print("=" * 60)
print(f"🧪 CHARGEMENT DU MODÈLE MERGÉ")
print("=" * 60)

model_test, tokenizer_test = FastLanguageModel.from_pretrained(
    model_name    = MERGED_DIR,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = torch.float16,
    load_in_4bit  = False,
)
FastLanguageModel.for_inference(model_test)
print("✅ Modèle chargé et prêt !")

# ── Rappel du dataset utilisé ────────────────────────────────
print("\n📊 Dataset d'entraînement utilisé :")
df_bal = pd.read_csv(BALANCED_PATH)
print(df_bal['language'].value_counts().to_string())
print(f"   Total : {len(df_bal)} exemples équilibrés")

# ── Fonction de génération ────────────────────────────────────
def generate_response(messages: list,
                       max_new_tokens: int = 200,
                       temperature: float  = 0.7,
                       top_p: float        = 0.9) -> str:
    input_text = tokenizer_test.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_test(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_test.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = temperature,
            top_p          = top_p,
            do_sample      = True,
            pad_token_id   = tokenizer_test.eos_token_id,
        )

    response = tokenizer_test.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🧪 CHARGEMENT DU MODÈLE MERGÉ
==((====))==  Unsloth 2026.3.8: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/fine tuning/model_merged_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/fine tuning/model_merged_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Modèle chargé et prêt !

📊 Dataset d'entraînement utilisé :
language
moroccan_darija    842
english            842
french             842
arabic             842
   Total : 3368 exemples équilibrés


In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CAS DE TEST — 1 test par langue du dataset équilibré       ║
# ╚══════════════════════════════════════════════════════════════╝
test_cases = [

    # ── 🇲🇦 MOROCCAN DARIJA ──────────────────────────────────
    {
        "label"   : "🇲🇦 Darija — Salutation basique",
        "langue"  : "moroccan_darija",
        "messages": [
            {"role": "system",    "content": "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."},
            {"role": "user",      "content": "سلام، كيداير؟"},
        ]
    },
    {
        "label"   : "🇲🇦 Darija — Conseil pratique",
        "langue"  : "moroccan_darija",
        "messages": [
            {"role": "system",    "content": "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."},
            {"role": "user",      "content": "شنو أحسن طريقة باش نتعلم البرمجة من الصفر؟"},
        ]
    },
    {
        "label"   : "🇲🇦 Darija — Multi-turn (وصفة)",
        "langue"  : "moroccan_darija",
        "messages": [
            {"role": "system",    "content": "نت مساعد مفيد. كتجاوب بالدارجة المغربية."},
            {"role": "user",      "content": "عطيني وصفة ديال الحريرة"},
            {"role": "assistant", "content": "واخا! الحريرة محتاجة: طماطم، عدس، حمص، لحم، كزبرة، معدنوس، ومخلطة. كبدا بالزيت والبصل مع اللحم..."},
            {"role": "user",      "content": "وشحال من وقت خاصها تطيب؟"},
        ]
    },

    # ── 🇸🇦 ARABIC (MSA) ──────────────────────────────────────
    {
        "label"   : "🇸🇦 Arabic — سؤال عام",
        "langue"  : "arabic",
        "messages": [
            {"role": "system",    "content": "أنت مساعد ذكي ومفيد. تتحدث باللغة العربية الفصحى."},
            {"role": "user",      "content": "ما هي أفضل طريقة لتعلم لغة جديدة؟"},
        ]
    },
    {
        "label"   : "🇸🇦 Arabic — نصيحة تقنية",
        "langue"  : "arabic",
        "messages": [
            {"role": "system",    "content": "أنت مساعد تقني متخصص. تجيب باللغة العربية."},
            {"role": "user",      "content": "كيف أحمي حسابي على الإنترنت من الاختراق؟"},
        ]
    },

    # ── 🇫🇷 FRANÇAIS ─────────────────────────────────────────
    {
        "label"   : "🇫🇷 Français — Question générale",
        "langue"  : "french",
        "messages": [
            {"role": "system",    "content": "Tu es un assistant utile et bienveillant. Tu réponds en français."},
            {"role": "user",      "content": "Quelle est la capitale du Maroc et pourquoi est-elle importante ?"},
        ]
    },
    {
        "label"   : "🇫🇷 Français — Conseil professionnel",
        "langue"  : "french",
        "messages": [
            {"role": "system",    "content": "Tu es un assistant utile. Tu réponds en français."},
            {"role": "user",      "content": "Comment rédiger un bon email professionnel ?"},
        ]
    },

    # ── 🇬🇧 ENGLISH ──────────────────────────────────────────
    {
        "label"   : "🇬🇧 English — Practical question",
        "langue"  : "english",
        "messages": [
            {"role": "system",    "content": "You are a helpful and knowledgeable assistant."},
            {"role": "user",      "content": "How can I improve my productivity when working from home?"},
        ]
    },
    {
        "label"   : "🇬🇧 English — Tech advice",
        "langue"  : "english",
        "messages": [
            {"role": "system",    "content": "You are a helpful assistant specialized in technology."},
            {"role": "user",      "content": "What are the best practices for keeping my social media accounts secure?"},
        ]
    },
]

# ╔══════════════════════════════════════════════════════════════╗
# ║  EXÉCUTION DES TESTS PAR LANGUE                             ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("🧪 RÉSULTATS DES TESTS — PAR LANGUE")
print("=" * 60)

langue_actuelle = ""
scores = {"moroccan_darija": [], "arabic": [], "french": [], "english": []}

for i, test in enumerate(test_cases, 1):

    # Séparateur par langue
    if test['langue'] != langue_actuelle:
        langue_actuelle = test['langue']
        print(f"\n{'━' * 60}")
        print(f"  Langue : {langue_actuelle.upper()}")
        print(f"{'━' * 60}")

    print(f"\n[Test {i}] {test['label']}")
    print("-" * 50)

    last_user_msg = next(
        (m['content'] for m in reversed(test['messages']) if m['role'] == 'user'), ""
    )
    print(f"👤 User     : {last_user_msg}")

    response = generate_response(test['messages'])
    print(f"🤖 Assistant: {response}")

    # Évaluation simple : réponse non vide et > 10 chars
    is_valid = len(response) > 10
    scores[test['langue']].append(is_valid)
    print(f"   {'✅ Réponse valide' if is_valid else '❌ Réponse trop courte/vide'}")

# ╔══════════════════════════════════════════════════════════════╗
# ║  RÉSUMÉ PAR LANGUE                                          ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("📊 RÉSUMÉ PAR LANGUE")
print("=" * 60)

for lang, results in scores.items():
    if results:
        taux = sum(results) / len(results) * 100
        barre = "█" * int(taux / 10) + "░" * (10 - int(taux / 10))
        print(f"  {lang:20s} : [{barre}] {taux:.0f}%  ({sum(results)}/{len(results)} OK)")

print("\n" + "=" * 60)
print("🎉 TESTS TERMINÉS !")
print(f"   Modèle   : {MERGED_DIR}")
print(f"   Dataset  : {BALANCED_PATH}")
print("=" * 60)

# ╔══════════════════════════════════════════════════════════════╗
# ║  BONUS — Test interactif (tape ta propre question)          ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("💬 TEST INTERACTIF — Entre ta propre question")
print("   (tape 'quit' pour arrêter)")
print("=" * 60)

SYSTEM_PROMPTS = {
    "1": ("🇲🇦 Darija",   "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."),
    "2": ("🇸🇦 Arabic",   "أنت مساعد ذكي ومفيد. تتحدث باللغة العربية."),
    "3": ("🇫🇷 Français", "Tu es un assistant utile. Tu réponds en français."),
    "4": ("🇬🇧 English",  "You are a helpful assistant."),
}

print("\nChoisis une langue :")
for k, (label, _) in SYSTEM_PROMPTS.items():
    print(f"  [{k}] {label}")

choix = input("\nTon choix (1/2/3/4) : ").strip()
if choix not in SYSTEM_PROMPTS:
    choix = "1"

label_choix, system_prompt = SYSTEM_PROMPTS[choix]
print(f"\n✅ Langue sélectionnée : {label_choix}")
print("   (tape 'quit' pour quitter)\n")

conversation = [{"role": "system", "content": system_prompt}]

while True:
    user_input = input("👤 Toi : ").strip()
    if user_input.lower() in ['quit', 'exit', 'q', 'خروج', 'sortir']:
        print("👋 Au revoir / بسلامة / Goodbye!")
        break
    if not user_input:
        continue

    conversation.append({"role": "user", "content": user_input})
    response = generate_response(conversation, max_new_tokens=300)
    print(f"🤖 Assistant : {response}\n")
    conversation.append({"role": "assistant", "content": response})


🧪 RÉSULTATS DES TESTS — PAR LANGUE

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Langue : MOROCCAN_DARIJA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Test 1] 🇲🇦 Darija — Salutation basique
--------------------------------------------------
👤 User     : سلام، كيداير؟


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

🤖 Assistant: أهلاً وسهلاً بك، كيف يمكنني مساعدتك اليوم؟
   ✅ Réponse valide

[Test 2] 🇲🇦 Darija — Conseil pratique
--------------------------------------------------
👤 User     : شنو أحسن طريقة باش نتعلم البرمجة من الصفر؟


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant: إحنا ممكن نبدأ بتعلم الأساسيات، مثل الحسابات الرياضية وطرق الفك.cracking.
   ✅ Réponse valide

[Test 3] 🇲🇦 Darija — Multi-turn (وصفة)
--------------------------------------------------
👤 User     : وشحال من وقت خاصها تطيب؟


...

🤖 Assistant : لما تفكر في القراية، حسبي أكتر من أن تأكل الأرز والتوت والطماطم والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب الكاملة والخضروات الحمراء واللحمة والجزرة والبصل والثوم والقرفة والخضروات الخضراء والحبوب

👤 Toi : نصائح باش نكون منظم فحياتي


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : لما تحدد هواياتك، حسبي تحدد مواعيد لكيفية البدء والانتهاء منها. تذكري أن تأخذ وقتاً للاسترخاء والراحة بعد كل يوم من الراحة. حسبي تفكر في كيفية تطوير هواياتك وزيادة مهاراتك فيها.

👤 Toi : fr


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : أحسن، حسيب تواصل مع عائلتك وحدهم أو مع صديقين لتشجيع بعضهم البعض على البقاء على هواياتهم. احترام المعايير الصحية للجسم والروح، حسبي تأخذ تمارين رياضية مرتقبة وممارسة التأمل. تذكري أن تبقى مستعداً للاختبارات والاختبارات. حسبي تفكر في إعداد جدول زمني واضح لحياتك اليومية والليالي.

👤 Toi : quit
👋 Au revoir / بسلامة / Goodbye!


In [16]:
print("\n" + "=" * 60)
print("💬 TEST INTERACTIF — Entre ta propre question")
print("   (tape 'quit' pour arrêter)")
print("=" * 60)

SYSTEM_PROMPTS = {
    "1": ("🇲🇦 Darija",   "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."),
    "2": ("🇸🇦 Arabic",   "أنت مساعد ذكي ومفيد. تتحدث باللغة العربية."),
    "3": ("🇫🇷 Français", "Tu es un assistant utile. Tu réponds en français."),
    "4": ("🇬🇧 English",  "You are a helpful assistant."),
}

print("\nChoisis une langue :")
for k, (label, _) in SYSTEM_PROMPTS.items():
    print(f"  [{k}] {label}")

choix = input("\nTon choix (1/2/3/4) : ").strip()
if choix not in SYSTEM_PROMPTS:
    choix = "1"

label_choix, system_prompt = SYSTEM_PROMPTS[choix]
print(f"\n✅ Langue sélectionnée : {label_choix}")
print("   (tape 'quit' pour quitter)\n")

conversation = [{"role": "system", "content": system_prompt}]

while True:
    user_input = input("👤 Toi : ").strip()
    if user_input.lower() in ['quit', 'exit', 'q', 'خروج', 'sortir']:
        print("👋 Au revoir / بسلامة / Goodbye!")
        break
    if not user_input:
        continue

    conversation.append({"role": "user", "content": user_input})
    response = generate_response(conversation, max_new_tokens=300)
    print(f"🤖 Assistant : {response}\n")
    conversation.append({"role": "assistant", "content": response})


💬 TEST INTERACTIF — Entre ta propre question
   (tape 'quit' pour arrêter)

Choisis une langue :
  [1] 🇲🇦 Darija
  [2] 🇸🇦 Arabic
  [3] 🇫🇷 Français
  [4] 🇬🇧 English

Ton choix (1/2/3/4) : 2

✅ Langue sélectionnée : 🇸🇦 Arabic
   (tape 'quit' pour quitter)

👤 Toi : من أنت؟


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : أنا مساعد ذكي مصمم للمساعدة في الإجابة على الأسئلة وتوفير المعلومات.

👤 Toi : كيف يمكنني الاستفادة منك؟


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : بالطبع، يمكنك طرح أي أسئلة لديك، سواء كانت حول الصحة العامة، التكنولوجيا، التاريخ، أو أي موضوع آخر.

👤 Toi : هل تستطيع مساعدتي في الدراسة؟


...

🤖 Assistant : لتعلم البرمجة من الصفر، يمكنك البدء بتعلم أساسيات اللغة البرمجية مثل HTML, CSS, JavaScript. يمكنك أيضاً استخدام مواقع مثل Codecademy أو Khan Academy للحصول على معرفة أكثر و深掘.

👤 Toi : كيف أبدأ مشروعًا صغيرًا؟


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : لبدء مشروع صغير، يجب عليك تحديد الهدف والتسويق، ثم البحث عن الفرص والدعم المالي المناسب. يمكنك أيضاً استخدام مواقع مثل Kickstarter أو Indiegogo للحصول على الدعم.

👤 Toi : كيف أتجنب التشتت؟


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : للحفاظ على التركيز، يمكنك استخدام برامج إدارة الوقت، مثل TimeOut أو Trello. يمكنك أيضاً مشاركة حياتك اليومية مع العائلة أو الأصدقاء لتعزيز الاهتمام بها.

👤 Toi : quit
👋 Au revoir / بسلامة / Goodbye!


In [17]:
print("\n" + "=" * 60)
print("💬 TEST INTERACTIF — Entre ta propre question")
print("   (tape 'quit' pour arrêter)")
print("=" * 60)

SYSTEM_PROMPTS = {
    "1": ("🇲🇦 Darija",   "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."),
    "2": ("🇸🇦 Arabic",   "أنت مساعد ذكي ومفيد. تتحدث باللغة العربية."),
    "3": ("🇫🇷 Français", "Tu es un assistant utile. Tu réponds en français."),
    "4": ("🇬🇧 English",  "You are a helpful assistant."),
}

print("\nChoisis une langue :")
for k, (label, _) in SYSTEM_PROMPTS.items():
    print(f"  [{k}] {label}")

choix = input("\nTon choix (1/2/3/4) : ").strip()
if choix not in SYSTEM_PROMPTS:
    choix = "1"

label_choix, system_prompt = SYSTEM_PROMPTS[choix]
print(f"\n✅ Langue sélectionnée : {label_choix}")
print("   (tape 'quit' pour quitter)\n")

conversation = [{"role": "system", "content": system_prompt}]

while True:
    user_input = input("👤 Toi : ").strip()
    if user_input.lower() in ['quit', 'exit', 'q', 'خروج', 'sortir']:
        print("👋 Au revoir / بسلامة / Goodbye!")
        break
    if not user_input:
        continue

    conversation.append({"role": "user", "content": user_input})
    response = generate_response(conversation, max_new_tokens=300)
    print(f"🤖 Assistant : {response}\n")
    conversation.append({"role": "assistant", "content": response})


💬 TEST INTERACTIF — Entre ta propre question
   (tape 'quit' pour arrêter)

Choisis une langue :
  [1] 🇲🇦 Darija
  [2] 🇸🇦 Arabic
  [3] 🇫🇷 Français
  [4] 🇬🇧 English

Ton choix (1/2/3/4) : 3

✅ Langue sélectionnée : 🇫🇷 Français
   (tape 'quit' pour quitter)

👤 Toi : Qui es-tu ?


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : Je suis un assistant virtuel créé par Anthropic pour être utile, honnête et authentique.

👤 Toi : Que fais-tu ?


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : J'ai été conçu pour aider à répondre aux questions, discuter de sujets divers et aider les utilisateurs à trouver des informations précises.

👤 Toi : bonjour cv


...

🤖 Assistant : Bien sûr, je peux vous aider avec vos études. Que voulez-vous savoir ou accomplir ?

👤 Toi : Donne-moi un plan pour apprendre la programmation à partir de zéro


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : D'accord, voici un plan simple pour apprendre la programmation à partir de zéro :

👤 Toi : Comment démarrer un petit projet ?


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : Pour démarrer un petit projet, voici quelques étapes que vous pouvez suivre :

👤 Toi : quit
👋 Au revoir / بسلامة / Goodbye!


In [ ]:
print("\n" + "=" * 60)
print("💬 TEST INTERACTIF — Entre ta propre question")
print("   (tape 'quit' pour arrêter)")
print("=" * 60)

SYSTEM_PROMPTS = {
    "1": ("🇲🇦 Darija",   "نت مساعد مفيد وودي. كتهضر بالدارجة المغربية."),
    "2": ("🇸🇦 Arabic",   "أنت مساعد ذكي ومفيد. تتحدث باللغة العربية."),
    "3": ("🇫🇷 Français", "Tu es un assistant utile. Tu réponds en français."),
    "4": ("🇬🇧 English",  "You are a helpful assistant."),
}

print("\nChoisis une langue :")
for k, (label, _) in SYSTEM_PROMPTS.items():
    print(f"  [{k}] {label}")

choix = input("\nTon choix (1/2/3/4) : ").strip()
if choix not in SYSTEM_PROMPTS:
    choix = "1"

label_choix, system_prompt = SYSTEM_PROMPTS[choix]
print(f"\n✅ Langue sélectionnée : {label_choix}")
print("   (tape 'quit' pour quitter)\n")

conversation = [{"role": "system", "content": system_prompt}]

while True:
    user_input = input("👤 Toi : ").strip()
    if user_input.lower() in ['quit', 'exit', 'q', 'خروج', 'sortir']:
        print("👋 Au revoir / بسلامة / Goodbye!")
        break
    if not user_input:
        continue

    conversation.append({"role": "user", "content": user_input})
    response = generate_response(conversation, max_new_tokens=300)
    print(f"🤖 Assistant : {response}\n")
    conversation.append({"role": "assistant", "content": response})


💬 TEST INTERACTIF — Entre ta propre question
   (tape 'quit' pour arrêter)

Choisis une langue :
  [1] 🇲🇦 Darija
  [2] 🇸🇦 Arabic
  [3] 🇫🇷 Français
  [4] 🇬🇧 English

✅ Langue sélectionnée : 🇬🇧 English
   (tape 'quit' pour quitter)



Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : hello! how can I help you today?

👤 Toi : hi


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Assistant : hi there! any particular topic you'd like to discuss?

